In [ ]:
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
from pytorch_lightning import seed_everything
from PIL import Image
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg

from captum.attr import visualization
from transformers import BertTokenizer
from models.main_net import main_net
from dataloader.utils import get_imgs

try:
    from IPython.display import display, HTML
    HAS_IPYTHON = True
except ImportError:
    HAS_IPYTHON = False
    
from constants import *
seed_everything(42)

In [ ]:
BASE_DIR = os.getcwd()
baseline_model_path = BASE_DIR+"/checkpoints/CoMA2/baseline/epoch=26-step=85238.ckpt"
CoMA_model_path = BASE_DIR+"/checkpoints/CoMA2/baseline+sentcesplit+conceptprototype+orth+prompotvector/epoch=25-step=82081.ckpt"

MAX_LENGTH = 112
device = "cuda" if torch.cuda.is_available() else "cpu"
start_layer_img =  10
start_layer_text =  -1

tokenizer = BertTokenizer.from_pretrained(
    os.path.join(BASE_DIR, './model_files/Bio_ClinicalBERT'))
baseline_model = main_net.load_from_checkpoint(baseline_model_path, strict=False).to(device)
CoMA_model = main_net.load_from_checkpoint(CoMA_model_path, strict=False).to(device)

baseline_model.text_encoder_q.model.encoder.layer[start_layer_text].attention.self.set_save_attention(True)
CoMA_model.text_encoder_q.model.encoder.layer[start_layer_text].attention.self.set_save_attention(True)

In [ ]:
def calculate(model, images, texts):
    img_feat_q, _, _ = model.img_encoder_q(
            images, model.propompt_vector, register_blk=True)
    img_emb_q = model.img_encoder_q.global_embed(img_feat_q)
    img_emb_q = F.normalize(img_emb_q, dim=-1)
    feature = None

    report_feat_q, _, _, _ = model.text_encoder_q(
        texts["input_ids"], texts["attention_mask"], texts["token_type_ids"])
    report_emb_q = model.text_encoder_q.global_embed(report_feat_q)
    report_emb_q = F.normalize(report_emb_q, dim=-1)

    logits_per_image = img_emb_q.mm(report_emb_q.t())
    logits_per_image /= 0.07
    logits_per_text = logits_per_image.transpose(0, 1)
    return logits_per_image, logits_per_text, feature

def interpret(image, texts, model, device, start_layer_img=start_layer_img, start_layer_text=start_layer_text):
    batch_size = texts['input_ids'].shape[0]
    logits_per_image, _, feature = calculate(model, image, texts)
    one_hot = np.zeros(logits_per_image.shape, dtype=np.float32)
    one_hot[torch.arange(batch_size), torch.arange(batch_size)] = 1
    one_hot = torch.from_numpy(one_hot).requires_grad_(True)
    one_hot = torch.sum(one_hot.cuda() * logits_per_image)
    model.zero_grad()

    image_attn_blocks = model.img_encoder_q.model.blocks

    if start_layer_img == -1:
        start_layer_img = len(image_attn_blocks) - 1

    num_tokens = image_attn_blocks[start_layer_img].attn.attention_map.shape[-1]
    R = torch.eye(num_tokens, num_tokens, dtype=image_attn_blocks[start_layer_img].attn.attention_map.dtype).to(device)
    R = R.unsqueeze(0).expand(batch_size, num_tokens, num_tokens)
    for i, blk in enumerate(image_attn_blocks):
        if i < start_layer_img:
            continue
        grad = torch.autograd.grad(one_hot, [blk.attn.attention_map], retain_graph=True)[0].detach()
        cam = blk.attn.attention_map.detach()
        cam = cam.reshape(-1, cam.shape[-1], cam.shape[-1])
        grad = grad.reshape(-1, grad.shape[-1], grad.shape[-1])
        cam = grad * cam
        cam = cam.reshape(batch_size, -1, cam.shape[-1], cam.shape[-1])
        cam = cam.clamp(min=0).mean(dim=1)
        R = R + torch.bmm(cam, R)
    if model.propompt_vector is None:
        image_relevance = R[:, 0, 1:]
    else:
        image_relevance = R[:, 0, 1:-len(model.propompt_vector)]
    
    text_attn_blocks = model.text_encoder_q.model.encoder.layer
    if start_layer_text == -1:
      start_layer_text = len(text_attn_blocks) - 1

    num_tokens = text_attn_blocks[start_layer_text].attention.self.get_attention_map().shape[-1]
    R_text = torch.eye(num_tokens, num_tokens, dtype=text_attn_blocks[start_layer_text].attention.self.get_attention_map().dtype).to(device)
    R_text = R_text.unsqueeze(0).expand(batch_size, num_tokens, num_tokens)
    for i, blk in enumerate(text_attn_blocks):
        if i < start_layer_text:
          continue
        grad = torch.autograd.grad(one_hot, [blk.attention.self.get_attention_map()], retain_graph=True)[0].detach()
        cam = blk.attention.self.get_attention_map().detach()
        cam = cam.reshape(-1, cam.shape[-1], cam.shape[-1])
        grad = grad.reshape(-1, grad.shape[-1], grad.shape[-1])
        cam = grad * cam
        cam = cam.reshape(batch_size, -1, cam.shape[-1], cam.shape[-1])
        cam = cam.clamp(min=0).mean(dim=1)
        R_text = R_text + torch.bmm(cam, R_text)
    text_relevance = R_text
    return text_relevance, image_relevance
  
def show_heatmap_on_text(text, text_encoding, R_text,sentence_split=True):
    CLS_idx = torch.where(text_encoding == 101)[0].squeeze()
    SEP_idx = torch.where(text_encoding == 102)[0].squeeze()
    R_text = R_text[CLS_idx, CLS_idx+1:SEP_idx]
    if R_text.sum() == 0:
        R_text = torch.ones_like(R_text)/len(R_text)
    text_scores = R_text / R_text.sum()
    text_scores = text_scores.flatten()
    text_tokens=tokenizer(text)['input_ids'][1:-1]
    text_tokens_decoded=[tokenizer.decode([a]) for a in text_tokens]
    text_tokens_decoded = text_tokens_decoded[:MAX_LENGTH - 2]
    if any(['##' in t for t in text_tokens_decoded]):
        n_text_scores = []
        n_text_tokens_decoded = []
        for i,token in enumerate(text_tokens_decoded):
            if '##' in token and i != 0:
                n_text_scores[-1] += text_scores.tolist()[i]
                n_text_tokens_decoded[-1] += text_tokens_decoded[i].replace('##','')
            else:
                n_text_scores.append(text_scores.tolist()[i])
                n_text_tokens_decoded.append(text_tokens_decoded[i])
        text_scores = torch.tensor(n_text_scores).to(text_scores)
        text_tokens_decoded = n_text_tokens_decoded
    if sentence_split:
        n_text_scores = []
        n_text_tokens_decoded = []
        for i,token in enumerate(text_tokens_decoded):
            if i == 0:
                n_text_scores.append(text_scores.tolist()[i])
                n_text_tokens_decoded.append(text_tokens_decoded[i])
            else:
                n_text_scores[-1] += text_scores.tolist()[i]
                if not token in [',','.']:
                    n_text_tokens_decoded[-1] += ' ' + text_tokens_decoded[i]
                else:
                    n_text_tokens_decoded[-1] += text_tokens_decoded[i]
                    n_text_scores.append(0)
                    n_text_tokens_decoded.append('')
        text_scores = torch.tensor(n_text_scores).to(text_scores)
        text_tokens_decoded = n_text_tokens_decoded
    return [(s,[1-c,1,1-c]) for s,c in zip(text_tokens_decoded, text_scores.cpu().numpy().tolist())]

def get_texts(fplist):
    texts = {'patient':[], 'study':[], 'texts': []}
    for fpath in fplist:
        text = None
        patient = fpath.split('/')[-2]
        name,_ = os.path.splitext(fpath.split('/')[-1])
        name = name.split('_')[0]
        with open(os.path.join(BASE_DIR,fpath), 'r') as f:
            ori_text = f.readlines()
        for line in ori_text:
            if 'FINDINGS' in line:
                text = ''
                continue
            if len(ori_text) == 1:
                text = ''
            if line.isspace() or text is None or 'IMPRESSION' in line:
                continue
            text += line.replace('\n','')
        
        texts['patient'].append(patient)
        texts['study'].append(name)
        texts['texts'].append(text)
    return texts

class visDataTransforms(object):
    def __init__(self, crop_size: int = 224):
        data_transforms = [
            transforms.CenterCrop(crop_size),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ]

        self.data_transforms = transforms.Compose(data_transforms)

    def __call__(self, image):
        return self.data_transforms(image)

def get_images(imgs_path):
    imgs = []
    for img_path in imgs_path:
        imgs.append(get_imgs(img_path, 256, visDataTransforms(224)).to(device))
    return torch.stack(imgs)
        

In [ ]:
def get_cam_images(image_relevances, image):
    # create heatmap from mask on image
    def show_cam_on_image(img, mask):
        heatmap = cv2.applyColorMap(np.uint8(255 * mask), cv2.COLORMAP_JET)
        heatmap = np.float32(heatmap) / 255
        cam = heatmap + np.float32(img)
        cam = cam / np.max(cam)
        return cam
    
    image = image.permute(1, 2, 0).data.cpu().numpy()
    image = (image - image.min()) / (image.max() - image.min())
    cam_images = []
    for image_relevance in image_relevances:
        dim = int(image_relevance.numel() ** 0.5)
        image_relevance = image_relevance.reshape(1, 1, dim, dim)
        image_relevance = torch.nn.functional.interpolate(image_relevance, size=224, mode='bilinear')
        image_relevance = image_relevance.reshape(224, 224).cuda().data.cpu().numpy()
        image_relevance = (image_relevance - image_relevance.min()) / (image_relevance.max() - image_relevance.min())
        vis = show_cam_on_image(image, image_relevance)
        vis = np.uint8(255 * vis)
        vis = cv2.cvtColor(np.array(vis), cv2.COLOR_RGB2BGR)
        cam_images.append(vis)
    
    return cam_images
    
def add_wrapped_color_text(ax, text_segments, start_pos, max_line_width, fontsize=12):
    canvas = FigureCanvasAgg(ax.figure)
    renderer = canvas.get_renderer()
    
    x, y = start_pos
    line_height = fontsize * 0.004
    current_line_width = 0
    
    for text, color in text_segments:
        words = text.split()
        for word in words:
            temp_text = ax.text(
                x, y, word + ' ', 
                ha='left', va='center',
                transform=ax.transAxes,
                fontsize=fontsize,
                bbox={
                    'facecolor': 'white',
                    'edgecolor': 'none',
                    'pad': 0,
                    'alpha': 0,
                    'boxstyle': 'Round,pad=0'
                })
            
            bb = temp_text.get_window_extent(renderer=renderer)
            bb_ax = bb.transformed(ax.transAxes.inverted())
            word_width = bb_ax.x1 - bb_ax.x0
            temp_text.remove()
            
            if current_line_width + word_width > max_line_width:
                y -= line_height
                x = start_pos[0]
                current_line_width = 0
                
            t = ax.text(
                x, y, word + ' ', 
                ha='left', va='center',
                transform=ax.transAxes,
                fontsize=fontsize,
                bbox={
                    'facecolor': color,
                    'edgecolor': 'none',
                    'pad': 0,
                    'boxstyle': 'Round,pad=0'
                })
            x += word_width
            current_line_width += word_width
    ax.axis('off')

def print_text_visual_bar(vis_data_records):
    dom = ["<table width: 100%>"]
    rows = [
        "<th style='text-align: center'>Word Importance</th>"
    ]
    for datarecord in vis_data_records:
        rows.append(
            "".join(
                [
                    "<tr>",
                    visualization.format_word_importances(
                        datarecord.raw_input_ids, datarecord.word_attributions
                    ),
                    "<tr>",
                ]))
    dom.append('<div style="border-top: 1px solid; margin-top: 5px; \
                padding-top: 5px; display: inline-block">')
    dom.append("<b>Legend: </b>")
    for value, label in zip([-1, 0, 1], ["Negative", "Neutral", "Positive"]):
        dom.append(
            '<span style="display: inline-block; width: 10px; height: 10px; \
            border: 1px solid; background-color: \
            {value}"></span> {label}  '.format(
                value=visualization._get_color(value), label=label
            )
        )
    dom.append("</div>")
    dom.append("".join(rows))
    dom.append("</table>")
    html = HTML("".join(dom))
    display(html)

    return html


In [ ]:
patients = [
    "p10002428/s50027225/e8c44648-ff02beea-3d5ff638-dec79b01-7df71a69.jpg",
    "p10012261/s56872341/edeabd5b-139705f9-7fdef9a3-c6bdb0ed-000cef8f.jpg",
    "p10013653/s53212555/3b9f9742-adbf9136-b41b8572-ed58df70-c59d06da.jpg"
    ]

imgs_path = [os.path.join("/mnt/E/dataset/MIMIC-CXR-JPG/physionet.org/files/mimic-cxr-jpg"+"/2.0.0/files/p10", p) for p in patients]

reports_path = [img_path.replace('/'+img_path.split('/')[-1],'_ccd.txt') for img_path in imgs_path]

n= len(patients)

imgs = get_images(imgs_path)
texts = get_texts(fplist=reports_path)['texts']

text = tokenizer(texts, return_tensors="pt", truncation=True, padding="max_length", max_length=MAX_LENGTH)
text = text.to(device)
_, b_R_image = interpret(model=baseline_model, image=imgs, texts=text, device=device)
c_R_text, c_R_image = interpret(model=CoMA_model, image=imgs, texts=text, device=device)
textid = text['input_ids']
batch_size = textid.shape[0]

plt.rcParams['figure.figsize'] = (18, 16)

fig, axs = plt.subplots(batch_size, 4, figsize=(24, 16))

for i in range(batch_size):
    text_segments = show_heatmap_on_text(texts[i], textid[i], c_R_text[i])
    cam_images = get_cam_images([b_R_image[i],c_R_image[i]], imgs[i])
    
    axs[i][0].imshow(Image.open(imgs_path[i]))
    axs[i][0].axis('off')
    
    for j,cam_img in enumerate(cam_images):
        axs[i][j+1].imshow(cam_img)
        axs[i][j+1].axis('off')
    
    add_wrapped_color_text(
        ax=axs[i][3],
        text_segments=text_segments,
        start_pos=(0.05, 0.95),
        max_line_width=0.95,
        fontsize=16
    )

axs[0][0].set_title('original image', fontsize=20)
axs[0][1].set_title('baseline', fontsize=20)
axs[0][2].set_title('CoMA', fontsize=20)
axs[0][3].set_title('Image report', fontsize=20)

plt.show()
plt.savefig(f'vis_{i}.png')